### Imports & Setup

In [1]:
from Bio import Entrez, SeqIO
Entrez.email = input("Enter your email address: ")

In [2]:
def find_in_entrez_db(db, query, n_results = 10):
    handle = Entrez.esearch(db=db, term=query, retmax=n_results)
    results = Entrez.read(handle)
    return results["IdList"]


def fetch_entrez_summary(db, ids):
    with Entrez.esummary(db=db, id=",".join(ids)) as handle:
        results = Entrez.read(handle)
        return results


def parse_entrez_summary(results, ids_map, keys_to_extract):
    fetched_genes_data = results["DocumentSummarySet"]["DocumentSummary"]
    parsed_genes_data = {}
    for entry, _id in zip(fetched_genes_data, ids_map):
       parsed_dict = {k: entry[k] for k in keys_to_extract}
       parsed_genes_data[_id] = parsed_dict
    return parsed_genes_data


def extract_result_info(records, keys):
    result = []
    for record in records:
        result.append({k: record[k] for k in keys})
    return result

In [23]:
def run_and_parse_query_gene_db(query, n_results = 10):
    result_ids = find_in_entrez_db("gene", query, n_results=n_results)
    summary_raw = fetch_entrez_summary("gene", result_ids)
    summary_parsed = parse_entrez_summary(summary_raw, result_ids, keys_to_extract=["Name", "Chromosome", "MapLocation", "Description"])
    return summary_parsed

results = run_and_parse_query_gene_db("brca1[gene]", n_results=10)
for gene_data in list(results.items())[:10]:
    print(gene_data)


('143892821', {'Name': 'BRCA1', 'Chromosome': 'Un', 'MapLocation': '', 'Description': 'breast cancer susceptibility1'})
('143825099', {'Name': 'BRCA1', 'Chromosome': '16', 'MapLocation': '', 'Description': 'BRCA1 DNA repair associated'})
('143765407', {'Name': 'BRCA1', 'Chromosome': '4', 'MapLocation': '', 'Description': 'BRCA1 DNA repair associated'})
('143687345', {'Name': 'BRCA1', 'Chromosome': '6', 'MapLocation': '', 'Description': 'BRCA1 DNA repair associated'})
('143409333', {'Name': 'Brca1', 'Chromosome': '11', 'MapLocation': '', 'Description': 'BRCA1 DNA repair associated'})
('672', {'Name': 'BRCA1', 'Chromosome': '17', 'MapLocation': '17q21.31', 'Description': 'BRCA1 DNA repair associated'})
('12189', {'Name': 'Brca1', 'Chromosome': '11', 'MapLocation': '11 D', 'Description': 'breast cancer 1, early onset'})
('497672', {'Name': 'Brca1', 'Chromosome': '10', 'MapLocation': '10q31', 'Description': 'BRCA1, DNA repair associated'})
('403437', {'Name': 'BRCA1', 'Chromosome': '9', 'M

In [27]:
results = run_and_parse_query_gene_db("BRCA1[gene] AND Homo Sapiens[orgn]", n_results=100)
for gene_data in list(results.items())[:10]:
    print(gene_data)

('672', {'Name': 'BRCA1', 'Chromosome': '17', 'MapLocation': '17q21.31', 'Description': 'BRCA1 DNA repair associated'})


In [39]:
brca1_gene_id, brca1_gene_details = list(results.items())[0]

In [38]:
def find_link_between_dbs(dbfrom, dbto, gene_id, **kwargs):
    with Entrez.elink(dbfrom=dbfrom, db=dbto, id=gene_id, **kwargs) as handle:
        results = Entrez.read(handle)
        records_list = results[0]["LinkSetDb"][0]["Link"]
        return [record["Id"] for record in records_list]

In [33]:
related_ids = find_link_between_dbs("gene", "omim", brca1_gene_id)
omim_summary = fetch_entrez_summary("omim", related_ids)
for record in omim_summary:
    print(record["Title"])

FANCONI ANEMIA, COMPLEMENTATION GROUP S; FANCS
PANCREATIC CANCER, SUSCEPTIBILITY TO, 4; PNCA4
BREAST-OVARIAN CANCER, FAMILIAL, SUSCEPTIBILITY TO, 1; BROVCA1
BREAST CANCER
BRCA1 DNA REPAIR-ASSOCIATED PROTEIN; BRCA1


In [8]:
related_ids = find_link_between_dbs("gene", "protein", brca1_gene_id)
protein_summary = fetch_entrez_summary("protein", related_ids)
for record in protein_summary:
    print(record["Gi"])

IntegerElement(2909903173, attributes={})
IntegerElement(2909903171, attributes={})
IntegerElement(2703623458, attributes={})
IntegerElement(2703623456, attributes={})
IntegerElement(2703623454, attributes={})
IntegerElement(2703623452, attributes={})
IntegerElement(2701656703, attributes={})
IntegerElement(2616260805, attributes={})
IntegerElement(2616260803, attributes={})
IntegerElement(2616260801, attributes={})
IntegerElement(2616260799, attributes={})
IntegerElement(2616260797, attributes={})
IntegerElement(2607637247, attributes={})
IntegerElement(2607637245, attributes={})
IntegerElement(2607637243, attributes={})
IntegerElement(2607637241, attributes={})
IntegerElement(2557734836, attributes={})
IntegerElement(2557734834, attributes={})
IntegerElement(2557734832, attributes={})
IntegerElement(2557734830, attributes={})
IntegerElement(2557734828, attributes={})
IntegerElement(2557734826, attributes={})
IntegerElement(2468373407, attributes={})
IntegerElement(2468373377, attribu

In [37]:
protein_id = 121949022

with Entrez.efetch(db="protein", id=protein_id, rettype="gb", retmode="text") as handle:
    record = SeqIO.read(handle, "genbank")
    print(record.name)
    print(record.description)
    print(record.seq)

Q3LRJ6_HUMAN
Breast cancer 1 early onset
MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLKLLNQKKGPSQCPLCKNDITKRSLQESTRFSQLVEELLKIICAFQLDTGLEYANSYNFAKKENNSPEHLKDEVSIIQSMGYRNRAKRLLQSEPENPSLQETSLSVQLSNLGTVRTLRTKQRIQPQKTSVYIELGSDSSEDTVNKATYCSVGDQELLQITPQGTRDEISLDSAKKAACEFSETDVTNTEHHQPSNNDLNTTEKRAAERHPEKYQGSSVSNLHVEPCGTNTHASSLQHENSSLLLTKDRMNVEKAEFCNKSKQPGLARSQHNRWAGSKETCNDRRTPSTEKKVDLNADPLCERKEWNKQKLPCSENPRDTEDVPWITLNSSIQKVNEWFSRSDELLGSDDSHDGESESNAKVADVLDVLNEVDEYSGSSEKIDLLASDPHEALICKSERVHSKSVESNIEDKIFGKTYRKKASLPNLSHVTENLIIGAFVTEPQIIQERPLTNKLKRKRRPTSGLHPEDFIKKADLAVQKTPEMINQGTNQTEQNGQVMNITNSGHENKTKGDSIQNEKNPNPIESLEKESAFKTKAEPISSSISNMELELNIHNSKAPKKNRLRRKSSTRHIHALELVVSRNLSPPNCTELQIDSCSSSEEIKKKKYNQMPVRHSRNLQLMEGKEPATGAKKSNKPNEQTSKRHDSDTFPELKLTNAPGSFTKCSNTSELKEFVNPSLPREEKEEKLETVKVSNNAEDPKDLMLSGERVLQTERSVESSSISLVPGTDYGTQESISLLEVSTLGKAKTEPNKCVSQCAAFENPKGLIHGCSKDNRNDTEGFKYPLGHEVNHSRETSIEMEESELDAQYLQNTFKVSKRQSFAPFSNPGNAEEECATFSAHSGSLKKQSPKVTFECEQKEENQGKNESNIKPVQTVNITAGFPVVGQKDKPVDNAKCSIKGGSRFCLSSQFR

In [47]:
related_ids = find_link_between_dbs("gene", "snp", brca1_gene_id, term="homo sapiens")
snp_summary = fetch_entrez_summary("snp", related_ids[:50])
summary_parsed = parse_entrez_summary(snp_summary, result_ids, keys_to_extract=["SNP_ID", "SNP_CLASS", "GENES", "CHRPOS"])
for record in summary_parsed.values():
    print(f"RecordID: {record['SNP_ID']} | Class: {record['SNP_CLASS']} | Gene: {record['GENES']} | Position: {record['CHRPOS']}")


RecordID: 1491587915 | Class: delins | Gene: [{'NAME': 'BRCA1', 'GENE_ID': '672'}] | Position: 17:43100630
RecordID: 1491573730 | Class: ins | Gene: [{'NAME': 'BRCA1', 'GENE_ID': '672'}] | Position: 17:43100630
RecordID: 1491558507 | Class: delins | Gene: [{'NAME': 'BRCA1', 'GENE_ID': '672'}] | Position: 17:43061586
RecordID: 80357783 | Class: delins | Gene: [{'NAME': 'BRCA1', 'GENE_ID': '672'}, {'NAME': 'NBR2', 'GENE_ID': '10230'}] | Position: 17:43124030
RecordID: 1491538523 | Class: ins | Gene: [{'NAME': 'BRCA1', 'GENE_ID': '672'}] | Position: 17:43066834
RecordID: 1491529521 | Class: ins | Gene: [{'NAME': 'BRCA1', 'GENE_ID': '672'}] | Position: 17:43100618
RecordID: 781716310 | Class: delins | Gene: [{'NAME': 'BRCA1', 'GENE_ID': '672'}] | Position: 17:43048519
RecordID: 1491496749 | Class: del | Gene: [{'NAME': 'BRCA1', 'GENE_ID': '672'}] | Position: 17:43100595
RecordID: 36085989 | Class: delins | Gene: [{'NAME': 'BRCA1', 'GENE_ID': '672'}] | Position: 17:43102949
RecordID: 132549

### Wnioski

Udało mi się uzyskać następujące informacje na temat genu BRCA1:
- Gen ten występuje nie tylko u ludzi. W pierwszym query otrzymałem wyniki wyszuiwania dla wielu innych organizmów. Zawężenie zapytania do gatunku Homo Sapiens pomogło mi zlokalizować gen, który mnie interesuje.
- Gen produkuje białko odpowiedzialne za naprawę DNA, hamuje rozwój nowotworów.
- Sądząc po wynikach wyszukiwania powiązanych sekwencji aminokwasowych gen ten koduje kilkadziesiąt takich sekwencji
- Q3LRJ6_HUMAN jest białkiem którego obecność wskazuje na możliwość występowania raka piersi u badanej osoby
- Baza danych snp zawiera mutacje genów. Wyniki wyszukiwania wskazują, że znane ludziom jest wiele mutacji tego genu o różnych typach i lokalizacjach